# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from tools import get_gemini_client
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [49]:
# constants
OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

MODEL_DICT = {
    'gpt': {'gpt-4o-mini', 'gpt-5-nano', 'gpt-5-mini', 'gpt-5.1'},
    'ollama': {'llama3.2', 'deepseek-r1:1.5b'},
    'gemini': {'gemini-3-pro-preview', 'gemini-flash-latest'}
}

def generic_llm_query(system_prompt, user_prompt, model='gemini-flash-latest', stream = False, json = False, output_response = False):
    if model in MODEL_DICT['gpt']:
        llm = OpenAI()
    elif model in MODEL_DICT['ollama']:
        llm = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    elif model in MODEL_DICT['gemini']:
        llm = OpenAI(
        base_url=GEMINI_BASE_URL,   
        api_key=GOOGLE_API_KEY,
    )
    else:
        raise ValueError(f"Model {model} not supported")
    # Create arguments dictionary
    if json:
        system_prompt += ' Answer in strict json format.'
    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    }
    # Conditionally add parameters
    if json:
        kwargs["response_format"] = {"type": "json_object"}
    if stream:
        kwargs["stream"] = True
    response = llm.chat.completions.create(**kwargs)

    # display stream as markdown
    if stream:
        output = ""
        if output_response:
            display_handle = display(Markdown(output), display_id=True)
        for chunk in response:
            content = chunk.choices[0].delta.content
            if content:
                output += content
                if output_response:
                    display_handle.update(Markdown(output))
        return output
    # display without stream
    if output_response:
        if json:
            pretty_print_json(response.choices[0].message.content)
        else:
            display(Markdown(response.choices[0].message.content))
    return response.choices[0].message.content

def pretty_print_json(data):
    """
    Prints JSON data in a readable, indented format with syntax highlighting.
    Accepts a dictionary, list, or a JSON string.
    """
    try:
        # If input is a string, try to parse it as JSON first
        if isinstance(data, str):
            data = json.loads(data)
        
        # Convert back to string with indentation
        pretty_json = json.dumps(data, indent=2, ensure_ascii=False)
        
        # Display using Markdown for syntax highlighting in the notebook
        display(Markdown(f"```json\n{pretty_json}\n```"))
        
    except json.JSONDecodeError:
        print("Invalid JSON string provided.")
    except Exception as e:
        print(f"Error prettifying JSON: {e}")

In [ ]:
from typing import List, Tuple
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

MODEL_DICT = {
    'gpt': {'gpt-4o-mini', 'gpt-5-nano', 'gpt-5-mini', 'gpt-5.1'},
    'ollama': {'llama3.2', 'deepseek-r1:1.5b'},
    'gemini': {'gemini-3-pro-preview', 'gemini-flash-latest'}
}

def pretty_print_json(data):
    """
    Prints JSON data in a readable, indented format with syntax highlighting.
    Accepts a dictionary, list, or a JSON string.
    """
    try:
        # If input is a string, try to parse it as JSON first
        if isinstance(data, str):
            data = json.loads(data)
        
        # Convert back to string with indentation
        pretty_json = json.dumps(data, indent=2, ensure_ascii=False)
        
        # Display using Markdown for syntax highlighting in the notebook
        display(Markdown(f"```json\n{pretty_json}\n```"))
        
    except json.JSONDecodeError:
        print("Invalid JSON string provided.")
    except Exception as e:
        print(f"Error prettifying JSON: {e}")

class LLMQuery:
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    def __init__(self,system_prompt, model='gemini-flash-latest', stream = False, json_format = False):
        self.model = model
        self.stream = stream
        self.json_format = json_format
        self.client = self.get_client()
        self.system_prompt = system_prompt
        self.chat_history = []
        self.response = ''

    def get_client(self):
        if self.model in MODEL_DICT['gpt']:
            client = OpenAI()
        elif self.model in MODEL_DICT['ollama']:
            client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
        elif self.model in MODEL_DICT['gemini']:
            client = OpenAI(
            base_url=GEMINI_BASE_URL,   
            api_key=GOOGLE_API_KEY,
        )
        else:
            raise ValueError(f"Model {self.model} not supported")
        return client
    
    def query(self, user_prompt, use_history = False, display_output = False, stream = None, json_format = None):
        if stream is None:
            stream = self.stream
        if json_format is None:
            json_format = self.json_format

        messages = [{"role": "system", "content": self.system_prompt}]
        if use_history:
            messages.extend(self.chat_history)
        messages.append({"role": "user", "content": user_prompt})
        kwargs = {
            "model": self.model,
            "messages": messages
        }
        if json_format:
            kwargs["response_format"] = {"type": "json_object"}
        if stream:
            kwargs["stream"] = True
        response = self.client.chat.completions.create(**kwargs)

        # stream output
        if stream:
            output = ""
            if display_output:
                display_handle = display(Markdown(output), display_id=True)
            for chunk in response:
                content = chunk.choices[0].delta.content
                if content:
                    output += content
                    if display_output:
                        display_handle.update(Markdown(output))
            self.response = output
        else:
            # without stream
            self.response = response.choices[0].message.content
            if display_output:
                self.display_response()

        self.chat_history.append({"role": "user", "content": user_prompt})
        self.chat_history.append({"role": "assistant", "content": self.response})
    
    def display_response(self):
        if self.json_format:
            pretty_print_json(self.response)
        else:
            display(Markdown(self.response))
    
    def get_chat_history_as_string(self):
        history = []
        for msg in self.chat_history:
            role = msg['role'].capitalize()
            content = msg['content']
            if role == "User":
                history.append(f"**User**: {content}")
            elif role == "Assistant":
                history.append(f"**Assistant**: {content}")
        return "\n\n".join(history)
    
    def display_chat_history(self):
        display(Markdown(self.get_chat_history_as_string()))
            

In [54]:
c = LLMQuery(system_prompt="You are a helpful assistant", stream=True)
c.query("Hello, my name is Martin", display_output=True)

Hello Martin, how can I help you today?

In [55]:
c.chat_history

[{'role': 'user', 'content': 'Hello, my name is Martin'},
 {'role': 'assistant', 'content': 'Hello Martin, how can I help you today?'}]

In [58]:
c.query("Whats my name?", use_history=True)
Markdown(c.get_chat_history_as_string())

**User**: Hello, my name is Martin

**Assistant**: Hello Martin, how can I help you today?

**User**: Whats my name?

**Assistant**: Your name is Martin.

**User**: Whats my name?

**Assistant**: As an AI, I don't have access to your personal information, so I don't know your name.

If you tell me your name, I'll be happy to remember it!

**User**: Whats my name?

**Assistant**: Your name is Martin. I remember you told me earlier!

In [16]:
display(Markdown(c.get_chat_history_as_string()))

User: Hello, my name is Martin


Hello Martin! It's a pleasure to meet you. How may I assist you today?

User: Whats my name?


Assistant: I don't have access to your personal information, including your name. Is there anything else I can help you with?

User: Hello, my name is Martin


Hello Martin! It's a pleasure to meet you. How may I assist you today?

User: Whats my name?


Assistant: I don't have access to your personal information, including your name. Is there anything else I can help you with?

User: Whats my name?


Your name is Martin, as you told me earlier.

In [6]:
# Simple Test
display(Markdown(generic_llm_query("You are a helpful assistant.", "Who are you?")))

I am a large language model, trained by Google.

In [36]:
# set up environment

SYSTEM_PROMPT_TECHNICAL_EXPERT = """
You are a technical expert with deep knowledge in software development and programming languages. You are able to answer questions about programming languages, frameworks, and tools. You are also able to answer questions about the latest trends in software development and programming languages.
"""

In [38]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [47]:
# Get gpt-4o-mini to answer, with streaming

expert = LLMQuery(system_prompt=SYSTEM_PROMPT_TECHNICAL_EXPERT, model='gpt-4o-mini', stream=True)
expert.query(question, display_output=True, stream = False)
# display in markdown

The code you provided uses Python's `yield from` statement in conjunction with a set comprehension. Let's break it down step by step:

1. **Set Comprehension**: 
   - The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension that iterates over a collection called `books`.
   - For each `book` in `books`, it calls `book.get("author")`. The `get` method is used to safely retrieve the value associated with the key `"author"` from each `book` dictionary. If the key does not exist, `get` returns `None` instead of raising an error.
   - The `if book.get("author")` condition filters the results so that only books with a non-None author are included in the resulting set. This means that it only collects authors from books that actually have an author entry.

2. **Yield from**:
   - The `yield from` statement is used to yield all values from the iterator that results from the set comprehension. In this case, it yields all the unique author values that were gathered in the set.
   - If the code is inside a generator function, this line will produce each author one by one whenever the generator is iterated over.

### Summary:
In summary, this line of code collects unique authors from a collection of book dictionaries (ignoring entries without authors) and yields them one by one. It effectively creates an iterable of authors from the provided `books` collection.

### Purpose:
The purpose of this code is to allow for more efficient handling of the various authors without accumulating them in an intermediate structure like a list, while also ensuring that each author is only reported once due to the nature of sets. This could be useful in scenarios such as processing a list of books in a library system, creating reports, or any task where you need the unique authors from a collection of book records.

In [40]:
expert.display_response()

**Assistant**: This single line of Python code is extremely dense, utilizing a **Set Comprehension** and the **`yield from`** keyword, indicating it is running inside a **Generator Function**.

Here is a detailed explanation of what the code does and why it is structured this way.

---

## 1. What the Code Does (The Process)

This code performs three distinct operations: filtering, de-duplication, and delegated yielding.

### Step 1: Set Comprehension (Filtering and De-duplication)

The core structure is a Python **Set Comprehension**:

```python
{book.get("author") for book in books if book.get("author")}
```

1.  **Iteration (`for book in books`):** It assumes `books` is an iterable (like a list) containing dictionaries, where each dictionary represents a book.
2.  **Filtering (`if book.get("author")`):** It checks the retrieved author value.
    *   `book.get("author")` safely tries to retrieve the value for the `"author"` key (returning `None` if the key is missing).
    *   In a boolean context, `None` or an empty string (`""`) evaluates to `False`. This ensures that only books that have a defined, non-empty author value are processed.
3.  **Collection (Set Creation):** The resulting values (the author names) are collected into a **Set** (indicated by the curly braces `{}`).

**Result of Step 1:** An entire Python `set` object is created in memory, containing every unique author name from the input `books` list that passed the filter.

### Step 2: Yield Delegation (`yield from`)

The `yield from` statement takes the set created in Step 1 and delegates iteration to it.

```python
yield from [the resulting set]
```

Instead of the function writing a loop like:

```python
for item in the_set:
    yield item
```

`yield from` handles this iteration internally and efficiently. It yields each unique author name, one by one, to the caller of the generator function.

## 2. Why it Works This Way (Purpose and Benefits)

This structure is used to achieve two primary goals concisely: **guaranteed uniqueness** and **generator delegation**.

### A. The Primary Goal: Guaranteed Uniqueness

The core reason for using the **Set Comprehension** (curly braces `{}`) is to efficiently guarantee that the output stream contains only unique author names.

*   If the input list `books` had 10 entries for "Jane Austen," the resulting set would only contain "Jane Austen" once.
*   Using a set comprehension is generally faster and more Pythonic for de-duplication than sorting a list and manually checking for duplicates.

### B. Efficiency and Clarity with `yield from`

`yield from` simplifies the code structure. It signifies that the generator function is delegating its responsibilities to iterate over another iterable (the set of authors). This provides a cleaner syntax compared to writing an explicit internal loop.

### C. Filtering and Safety

The use of `.get("author")` provides a safe way to access dictionary keys without raising a `KeyError` if the key is missing. The subsequent check ensures that the yielded results are meaningful (not `None` or an empty string).

## 3. Important Context: Materialization vs. Laziness

It is critical to note the difference between a Set Comprehension and a Generator Expression:

| Feature | Set Comprehension (`{...}`) | Generator Expression (`(...)`) |
| :--- | :--- | :--- |
| **Materialization** | **Eager.** The entire set is built in memory *before* yielding starts. | **Lazy.** Items are processed and yielded one at a time. |
| **Goal** | Efficient de-duplication (Uniqueness). | Memory efficiency and immediate start of processing. |

Because the code uses curly braces (`{...}`), the *entire* list of unique authors must be calculated and stored in memory as a set before the first result can be `yield`ed. This structure prioritizes **uniqueness** over strict **laziness** for this specific step. If uniqueness were not required, the efficient, lazy alternative would be a Generator Expression:

```python
# A lazy generator expression (no uniqueness guarantee)
yield from (book.get("author") for book in books if book.get("author")) 
```

In [32]:
# Get Llama 3.2 to answer
stream_response = generic_llm_query(SYSTEM_PROMPT_TECHNICAL_EXPERT, question, model='llama3.2', stream=False, output_reponse=True, json=True)

{
  "type": "Code Explanation",
  "description": "Declarative Code with Yield From",
  "code": "[{\"line\":1,\"code\":\"yield from {book.get(\\\"author\\\") for book in books if book.get(\\\"author\")}\\\\n\\",", ":", "  }